# Novice — Dataset Exploration
**GNU General Public License v3.0**  
Copyright (C) 2024 Jean Robert Gatwaza — African Leadership University

This notebook explores the CPR-Coach `Sample_Dataset` from Google Drive.

## Setup
1. Connect Google Drive in Claude (connector panel)
2. Copy `Sample_Dataset` to `ml_pipeline/data/raw/Sample_Dataset/`
3. Run `extract_landmarks.py` to generate `.npy` files
4. Then run this notebook

**Reference**: Wang et al. (2023) CPR-Coach — ICCV 2023 Demo

In [ ]:
import sys, os
sys.path.insert(0, '../src')
os.chdir('..')  # run from ml_pipeline/ root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from pathlib import Path
from collections import Counter

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded')
print(f"  landmark dir : {cfg['dataset']['landmarks_dir']}")
print(f"  raw video dir: {cfg['dataset']['raw_dir']}")

## 1. Raw dataset inventory

In [ ]:
raw_dir = Path(cfg['dataset']['raw_dir'])

# TODO: Mount your Sample_Dataset from Google Drive here
# After connecting Google Drive connector in Claude:
#   cp -r /path/to/gdrive/Sample_Dataset data/raw/

if not raw_dir.exists():
    print(f'⚠ Dataset not found at {raw_dir}')
    print('  Connect Google Drive and copy Sample_Dataset to data/raw/')
else:
    videos = list(raw_dir.rglob('*.mp4')) + list(raw_dir.rglob('*.avi'))
    print(f'Found {len(videos)} video files')
    
    # Class distribution
    labels = [v.parent.name for v in videos]
    counts = Counter(labels)
    
    df = pd.DataFrame([
        {'Class': k, 'Videos': v, 'Class Index': i}
        for i, (k, v) in enumerate(sorted(counts.items()))
    ])
    print('\nClass distribution:')
    display(df)

## 2. Landmark feature distribution

In [ ]:
lm_dir = Path(cfg['dataset']['landmarks_dir'])

if not lm_dir.exists():
    print(f'⚠ Landmarks not found at {lm_dir}')
    print('  Run: python src/data/extract_landmarks.py --config config.yaml')
else:
    all_feats = []
    all_labels = []
    
    class_map = {v: k for k, v in cfg['dataset']['classes'].items()}
    
    for npy_path in sorted(lm_dir.rglob('*.npy')):
        seq = np.load(str(npy_path))   # (T, 12)
        label = npy_path.parent.name
        all_feats.append(seq)
        all_labels.extend([label] * len(seq))
    
    if all_feats:
        X = np.vstack(all_feats)
        print(f'Total frames: {len(X):,}')
        print(f'Feature dims: {X.shape[1]}')
        print(f'\nFeature statistics:')
        
        feat_names = [
            'left_elbow_angle', 'right_elbow_angle', 'mean_elbow_angle',
            'spine_verticality', 'wrist_y', 'wrist_vel_y', 'wrist_acc_y',
            'norm_depth', 'shoulder_width', 'mean_confidence',
            'left_elbow_vis', 'right_elbow_vis'
        ]
        
        stats_df = pd.DataFrame({
            'Feature': feat_names,
            'Mean':  X.mean(axis=0).round(3),
            'Std':   X.std(axis=0).round(3),
            'Min':   X.min(axis=0).round(3),
            'Max':   X.max(axis=0).round(3),
        })
        display(stats_df)

## 3. Elbow angle distributions by class
Key diagnostic: `bent_elbows` (E05) should show clearly lower angles.

In [ ]:
# TODO: Run after landmarks are extracted
# This cell plots elbow angle distributions per class
# to verify that the feature is discriminative for E05 (bent_elbows)

if 'X' in dir() and len(X) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Feature 2 = mean_elbow_angle
    for ax, (feat_idx, feat_name) in zip(axes, [(0, 'Left elbow angle'), (2, 'Mean elbow angle')]):
        for label in sorted(set(all_labels)):
            mask = np.array(all_labels) == label
            ax.hist(X[mask, feat_idx], bins=40, alpha=0.5, label=label, density=True)
        ax.axvline(160, color='red', linestyle='--', label='Lock threshold (160°)')
        ax.set_xlabel('Angle (degrees)')
        ax.set_ylabel('Density')
        ax.set_title(feat_name)
        ax.legend(fontsize=7)
    
    plt.tight_layout()
    plt.savefig('notebooks/elbow_angle_distribution.png', dpi=150)
    plt.show()
    print('Saved: notebooks/elbow_angle_distribution.png')
else:
    print('Run landmark extraction first')

## 4. Wrist Y velocity — compression rate signal
Peaks in wrist_vel_y correspond to compression downstrokes.

In [ ]:
from scipy.signal import find_peaks

# TODO: load one 'correct_compression' sequence and show peaks
sample_path = next(iter(sorted(lm_dir.rglob('correct_compression/*.npy'))), None) if 'lm_dir' in dir() else None

if sample_path and sample_path.exists():
    seq = np.load(str(sample_path))   # (T, 12)
    wrist_y   = seq[:, 4]             # feature index 4 = wrist_y
    velocity  = seq[:, 5]             # feature index 5 = wrist_vel_y
    
    peaks, _ = find_peaks(velocity, distance=10, height=0.005)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    t = np.arange(len(wrist_y)) / 25   # seconds at 25 fps
    
    ax1.plot(t, wrist_y, color='#00E5A0', linewidth=1)
    ax1.set_ylabel('Wrist Y (normalized)')
    ax1.set_title('Wrist position — correct compression sequence')
    
    ax2.plot(t, velocity, color='#FFC947', linewidth=1)
    ax2.scatter(t[peaks], velocity[peaks], color='#FF4D6D', zorder=5, s=30, label='Peaks')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Velocity')
    ax2.set_title('Wrist velocity — compression peaks')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('notebooks/wrist_velocity_peaks.png', dpi=150)
    plt.show()
    
    if len(peaks) >= 2:
        intervals = np.diff(peaks) / 25
        bpm = 60 / np.mean(intervals)
        print(f'Detected peaks: {len(peaks)}')
        print(f'Estimated BPM: {bpm:.1f}  (should be 100–120 for correct class)')
else:
    print('No correct_compression .npy files found — run extraction first')